# Full SFT on DGX Spark: Fine-Tuning Granite 2B Without LoRA

**Algorithm:** Full Supervised Fine-Tuning (all parameters)  
**Model:** `ibm-granite/granite-3.3-2b-instruct`  
**Hardware:** NVIDIA DGX Spark (GB10, 130.7 GB unified memory)  
**Dataset:** `b-mc2/sql-create-context` (natural language → SQL)  

---

Full SFT updates **every parameter** in the model — no low-rank approximations, no freezing.
This is the gold standard for fine-tuning quality, but it's also the most memory-hungry approach.

For Granite 2B, the memory estimator says 44–57 GB. The DGX Spark has 130.7 GB of unified
memory, so we should have headroom. But SFT through Training Hub uses the
[instructlab-training](https://github.com/instructlab/training) backend, which expects
DeepSpeed — and DeepSpeed on ARM is an adventure.

This is a **moderate risk** experiment. Let's find out what happens.

## 1. DGX Spark System Profile

> **Estimated run time:** ~5 seconds

In [ ]:
import platform
import subprocess
import torch
import os

print("=" * 60)
print("DGX Spark System Profile")
print("=" * 60)

print(f"\nPlatform:    {platform.machine()}")
print(f"Processor:   {platform.processor() or 'ARM (Grace)'}")
print(f"OS:          {platform.platform()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    cc = torch.cuda.get_device_capability(0)
    print(f"\nGPU:         {gpu_name}")
    print(f"GPU Memory:  {gpu_mem:.1f} GB")
    print(f"Compute Cap: {cc[0]}.{cc[1]}")
    print(f"CUDA:        {torch.version.cuda}")
else:
    print("\nNo CUDA GPU detected!")

print(f"\nPyTorch:     {torch.__version__}")
print(f"BF16:        {torch.cuda.is_bf16_supported()}")

total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f"\nSystem RAM:  {total_ram:.1f} GB (shared with GPU on DGX Spark)")

## 2. Environment Check — DeepSpeed & Flash Attention

Training Hub's SFT backend (`instructlab-training`) uses DeepSpeed for distributed training
and optionally flash-attn for memory-efficient attention. Both have C++/CUDA extensions
that need to compile for our platform.

> **Estimated run time:** ~2 seconds

In [ ]:
import importlib.metadata

packages = [
    "training-hub", "instructlab-training", "mini-trainer",
    "peft", "bitsandbytes", "trl", "accelerate",
    "transformers", "datasets", "torch",
    "deepspeed", "flash-attn", "xformers",
]

print(f"{'Package':<25} {'Status':<15} {'Version'}")
print("-" * 55)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:<25} {'INSTALLED':<15} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:<25} {'MISSING':<15} —")

### 2a. Attempting to Install DeepSpeed

DeepSpeed is critical for `instructlab-training`'s SFT pipeline. It has CUDA kernels
that need to be compiled from source on non-standard platforms.

> **Expectation:** May require building from source with `DS_BUILD_OPS=0` to skip
> optional ops, or may fail entirely on ARM + sm_121.

> **Estimated run time:** 30–300 seconds (pip resolve + possible source build)

In [ ]:
%%time
# Attempt 1: Standard pip install
!pip install deepspeed 2>&1 | tail -20

In [ ]:
# Check if DeepSpeed works
try:
    import deepspeed
    print(f"DeepSpeed installed! Version: {deepspeed.__version__}")
    # Check which ops are available
    !ds_report 2>&1 | head -40
    DEEPSPEED_AVAILABLE = True
except ImportError as e:
    print(f"DeepSpeed import failed: {e}")
    DEEPSPEED_AVAILABLE = False

In [ ]:
# If standard install failed, try building with minimal ops
if not DEEPSPEED_AVAILABLE:
    print("Attempting DeepSpeed install with minimal ops...")
    !DS_BUILD_OPS=0 pip install deepspeed --no-build-isolation 2>&1 | tail -20

    try:
        import deepspeed
        print(f"\nDeepSpeed installed (minimal)! Version: {deepspeed.__version__}")
        DEEPSPEED_AVAILABLE = True
    except ImportError as e:
        print(f"\nStill failed: {e}")
        print("DeepSpeed is not available on this platform.")

In [ ]:
%%time
# Attempt flash-attn install
print("Attempting flash-attn install...")
!pip install flash-attn 2>&1 | tail -15

try:
    import flash_attn
    print(f"\nflash-attn installed! Version: {flash_attn.__version__}")
    FLASH_ATTN_AVAILABLE = True
except ImportError as e:
    print(f"\nflash-attn not available: {e}")
    print("Not critical — PyTorch SDPA will be used as fallback.")
    FLASH_ATTN_AVAILABLE = False

## 3. Memory Estimation — Will Full SFT Fit?

Full SFT on 2B requires storing all parameters, gradients, and optimizer states.
Let's check the numbers.

> **Estimated run time:** ~30–60 seconds (downloads model config from HuggingFace on first run)

In [ ]:
%%time
from training_hub import estimate

GPU_MEM_BYTES = int(130.7 * 1024**3)
MODEL = "ibm-granite/granite-3.3-2b-instruct"

print("=" * 60)
print("Memory Estimate: Full SFT — Granite 2B")
print("=" * 60)
sft_est = estimate(
    training_method="sft",
    num_gpus=1,
    gpu_memory=GPU_MEM_BYTES,
    model_path=MODEL,
    verbose=2,
)

In [ ]:
def fmt_gb(b):
    return f"{b / 1024**3:.1f} GB"

print("\nMemory Summary:")
print(f"  Available:      {fmt_gb(GPU_MEM_BYTES)}")
print(f"  SFT estimate:   {fmt_gb(sft_est[0])} – {fmt_gb(sft_est[2])}")
headroom = GPU_MEM_BYTES - sft_est[2]
print(f"  Headroom:       {fmt_gb(headroom)} (worst case)")
print(f"\nVerdict: {'Should fit with margin.' if headroom > 0 else 'TIGHT — may OOM!'}")

## 4. Dataset Preparation

Same dataset as the LoRA notebook — `sql-create-context` converted to JSONL messages format.

> **Estimated run time:** ~10–30 seconds (dataset download, or instant if cached from Notebook 1)

In [ ]:
%%time
from datasets import load_dataset
import json

ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(f"\nSample:")
print(json.dumps(ds[0], indent=2))

In [ ]:
SUBSET_SIZE = 2000
DATA_PATH = "data/sql_train.jsonl"

os.makedirs("data", exist_ok=True)

system_prompt = (
    "You are a SQL expert. Given a natural language question and the relevant table schema, "
    "write the correct SQL query."
)

with open(DATA_PATH, "w") as f:
    for i, row in enumerate(ds):
        if i >= SUBSET_SIZE:
            break
        user_content = f"Question: {row['question']}\n\nContext:\n{row['context']}"
        record = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": row["answer"]},
            ]
        }
        f.write(json.dumps(record) + "\n")

print(f"Wrote {min(SUBSET_SIZE, len(ds))} examples to {DATA_PATH}")
print(f"File size: {os.path.getsize(DATA_PATH) / 1024:.1f} KB")

# Preview
with open(DATA_PATH) as f:
    sample = json.loads(f.readline())
print(f"\nPreview:")
print(json.dumps(sample, indent=2))

## 5. Training Run — Full SFT with Training Hub

Training Hub's `sft()` function uses the `instructlab-training` backend. This is the same
pipeline used by the InstructLab project for full model fine-tuning.

Key considerations:
- **`max_tokens_per_gpu`**: Hard memory cap. We'll set this conservatively.
- **`nproc_per_node=1`**: Single GPU on DGX Spark.
- **DeepSpeed**: Required by the backend. If it's not installed, this will fail.

> **Estimated run time:** 15–45 minutes for full SFT on Granite 2B with 2000 examples
> (includes model download ~2 min if not cached, data processing ~2 min, training ~10–40 min).
> Full SFT is significantly slower than QLoRA due to updating all parameters.

In [ ]:
import time

CKPT_DIR = "checkpoints/sft_granite2b"
DATA_OUTPUT_DIR = "data/sft_processed"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

print("Starting Full SFT training...")
print(f"  Model:           {MODEL}")
print(f"  Data:            {DATA_PATH}")
print(f"  Checkpoint dir:  {CKPT_DIR}")
print(f"  DeepSpeed:       {'Available' if DEEPSPEED_AVAILABLE else 'NOT AVAILABLE — expect failure'}")
print()

In [ ]:
from training_hub import sft

start_time = time.time()

try:
    result = sft(
        model_path=MODEL,
        data_path=os.path.abspath(DATA_PATH),
        ckpt_output_dir=os.path.abspath(CKPT_DIR),
        data_output_dir=os.path.abspath(DATA_OUTPUT_DIR),
        # Training config — conservative for memory
        num_epochs=1,
        effective_batch_size=8,
        learning_rate=2e-5,
        max_seq_len=512,
        max_tokens_per_gpu=2048,  # Conservative memory cap
        warmup_steps=10,
        # Single GPU
        nproc_per_node=1,
    )
    elapsed = time.time() - start_time
    print(f"\nSFT training completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

except Exception as e:
    elapsed = time.time() - start_time
    print(f"\nSFT training failed after {elapsed:.1f}s")
    print(f"Error: {type(e).__name__}: {e}")
    print("\nSee the Troubleshooting section below.")
    import traceback
    traceback.print_exc()

## 6. GPU Memory Usage

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3

    print("Peak GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.1f} GB")
    print(f"  Headroom:  {total - reserved:.1f} GB")
else:
    print("No CUDA device available for memory stats.")

## 7. Troubleshooting Log

### Known Issues on DGX Spark for Full SFT

| Issue | Detail | Workaround |
|-------|--------|------------|
| DeepSpeed ARM | No pre-built wheels for aarch64 | `DS_BUILD_OPS=0 pip install deepspeed --no-build-isolation` |
| DeepSpeed CUDA ops | sm_121 not supported by DeepSpeed's CUDA kernels | Build from source with `TORCH_CUDA_ARCH_LIST="12.1"` |
| flash-attn | No ARM + Blackwell wheels | Use PyTorch SDPA instead |
| Single GPU + DeepSpeed | FSDP/ZeRO overhead on 1 GPU | May not distribute but still adds overhead |
| Memory pressure | 44-57 GB estimate for 2B | Use conservative `max_tokens_per_gpu` |

### What Worked
- *Fill in after running*

### What Didn't Work
- *Fill in after running*

### Key Observations
- *Fill in after running*

## 8. Results & Next Steps

### Summary
- **Algorithm:** Full SFT (all parameters)
- **Model:** Granite 2B
- **Hardware:** DGX Spark (GB10, 130.7 GB unified memory)
- **Training time:** *Fill in after running*
- **Peak memory:** *Fill in after running*
- **Status:** *Fill in after running*

### Comparison with LoRA
- Full SFT uses ~10x more memory than QLoRA
- Full SFT updates all parameters → potentially better quality
- Full SFT requires DeepSpeed → harder to get running on exotic hardware

### Navigation
- [← Notebook 1: LoRA+SFT](01_lora_sft_spark.ipynb)
- [Notebook 3: OSFT →](03_osft_spark.ipynb)